In [ ]:
from torch.util.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
from torchvision import transform as trn
import torch.nn as nn


: 

In [ ]:
class DataSet(Dataset):

    def __init__(self, csv_file, transform=None):
        self.data_df = pd.read_csv(csv_file)
        self.transform = transform
    
    def __len__(self):
        return len(self.data_df)
    
    def __getitem__(self, idx):
        img_path = self.data_df.iloc[idx, 0]
        label = self.data_df[idx, 1]
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            return self.__getitem__((idx + 1) % self.__len__()) # pre-clean or post-hoc collate
        if self.transform:
            img = self.transform(img)
        return img, label

def calc_stats():
    transform = trn.Compose([
        trn.Resize((256, 256)),
        trn.CenterCrop(224),
        trn.ToTensor(),
        # trn.Normalize(mean=[], std=[]),
    ])

    dataset = DataSet('a_csv.csv', transform=transform)
    dataloader = DataLoader(dataset, shuffle=False, batch_size=64)

    tot_sum = torch.zeros(3, dtype=torch.float64)
    tot_square_sum = torch.zeros(3, dtype=torch.float64)
    tot_pix = 0
    for data, _ in dataloader:
        bs, c, h, w = data.size()
        tot_sum += torch.sum(data, dim=[0, 2, 3]) # bs, c, h, w
        tot_square_sum += torch.sum(data ** 2, dim=[0, 2, 3]) # bs, c, h, w
        tot_pix += bs * h * w
    mean = tot_sum / tot_pix
    std = (tot_square_sum / tot_pix - mean ** 2) ** 0.5
    return mean, std

means, stds = calc_stats()

In [ ]:
class MyNet(nn.Module):

    def __init__(self, num_layers=3, ac_func=nn.ReLU):
        super().__init__()
        layers = []
        in_channels = 3
        out_channels = 16

        for i in range(num_layers):
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)) # bs, indim, h, w -> bs, outdim, h, w
            layers.append(nn.BatchNorm2d(out_channels))

            layers.append(ac_func())

            layers.append(nn.MaxPool2d(2)) # bs, outdim, h/2, w/2
            in_channels = out_channels
            out_channels *= 2

        self.feature_extractor = nn.Sequential(*layers) # bs, outdim * (2 ** (nlayer - 1)), h/(2 ** (nlayer - 1)), w/(2 ** (nlayer - 1))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), # bs, odim * 2**x, 1, 1
            nn.Flatten(), # bs, odim * 2**x
            nn.Linear(in_channels, 1), # bs, 1
            nn.sigmoid() # bs, 1 
        )

    def forward(self, x):
        return self.classifier(self.feature_extractor(x))


In [ ]:
BATCH_SIZE = 64
EPOCH = 5
LR = 1e-3

transform = trn.Compose([
        trn.Resize((256, 256)),
        trn.CenterCrop(224),
        trn.ToTensor(),
        # trn.Normalize(mean=[], std=[]),
    ])

trainset = Dataset('train.csv', transform)
trainloader = DataLoader(trainset, shuffle=True, batch_size=BATCH_SIZE)

model = MyNet()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = torch.nn.BCELoss()

model.train()
for epoch in range(EPOCH):
    gt = []
    pred = []
    loss = 0
    for data, label in trainloader:
        optimizer.zero_grad()

        output = model(data)
        l = criterion(output, label)
        l.backward()
        optimizer.step()

        gt.extend(label.float().unsqueeze(1).numpy())
        pred.extend((output > 0.5).float().detach().numpy())
        loss += l.item()
        
    f1 = metric.f1_score(gt, pred)
    print(f"Epoch {epoch}/{EPOCH}, Loss={loss/len(trainloader):.4f}, F1={f1:.4f}")

torch.save(model.state_dict(), "model.pth")

In [ ]:
model.eval()

testset = Dataset('test.csv', transform)
testloader = DataLoader(trainset, shuffle=True, batch_size=BATCH_SIZE)

results = []
with torch.no_grad():
    for data in testloader:
        output = model(data)
        results.extend(output.float().detach().numpy())
